# Layer scene prototype: LayerDiffuse SDXL scene-to-layer chain on a Kaggle T4

Runs the prototype's engine **inside this notebook** (section 6 can also serve it to the local UI):

1. background layer from its prompt
2. every element is drawn **into the scene below it** (`bg2ble`) and extracted as its own aligned RGBA layer (`bgble2fg`)
3. re-prompt one layer: regenerated into the layers below it, nothing else changes
4. story event: a new element generated into the current scene as a new top layer
5. move / resize layers (canvas transform, no model call), then the final stacked image

Settings: **Accelerator = GPU T4 x2**, **Internet = On**, dataset **layerproto-code** attached.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install "diffusers==0.37.0" "transformers>=4.51" accelerate safetensors peft einops opencv-python-headless

In [ ]:
import glob, json, os, shutil, sys, zipfile
src = glob.glob('/kaggle/input/**/layerproto_code.zip', recursive=True) + glob.glob('/kaggle/input/**/prototype/app.py', recursive=True)
assert src, 'Attach the layerproto-code dataset (layerproto_code.zip) first'
if src[0].endswith('.zip'):
    zipfile.ZipFile(src[0]).extractall('/kaggle/working')
else:
    shutil.copytree(os.path.dirname(src[0]), '/kaggle/working/prototype', dirs_exist_ok=True)
sys.path.insert(0, '/kaggle/working/prototype')
os.environ['HF_HUB_DISABLE_XET'] = '1'
print(os.listdir('/kaggle/working/prototype'))

In [ ]:
from layerproto.config import Settings
from layerproto.engine import Engine

S = Settings(
    image_backend='local', device='cuda', sdxl_model='stabilityai/stable-diffusion-xl-base-1.0',
    width=1024, height=1024, steps=20, guidance=7.0, alpha_quality='best',
    # add-ons off: no Ollama on Kaggle by default -> comma split, no captions
    step_refine=False, step_split=False, step_words=False, step_captions=False, step_word_check=False,
    variants_per_layer=1, scenes_dir='/kaggle/working/scenes')
eng = Engine(S)

In [ ]:
from IPython.display import display
from PIL import Image

def show(scene, title=''):
    folder = scene.folder(S.scenes_dir)
    tiles = [eng.compose(scene)] + [Image.open(f'{folder}/{l.current().file}') for l in eng._ordered(scene) if l.current()]
    W = 256
    grid = Image.new('RGBA', (W * len(tiles), W), (40, 160, 60, 255))
    for i, t in enumerate(tiles):
        tile = Image.new('RGBA', (W, W), (40, 160, 60, 255)); tile.alpha_composite(t.convert('RGBA').resize((W, W)))
        grid.paste(tile, (i * W, 0))
    print(title, '| final image, then each layer bottom -> top (green = transparent)')
    display(grid)

## 1. Describe the scene: first comma part(s) with scenery words = background, every other part = one element

In [ ]:
scene = eng.plan_scene('pixel art, night forest of tall pine trees, full moon and stars, '
                       'a large campfire with a stone ring in the center, a skeleton dancing on the right side of the fire')
for l in scene.layers: print(l.depth, l.role, '|', l.prompt)
print('style:', scene.style)

## 2. Generate: background, then each element drawn into the scene and extracted

In [ ]:
import time
t = time.time()
eng.generate_all(scene, progress=print)
print(f'{time.time() - t:.0f} s')
show(scene, 'generated')

## 3. Re-prompt one layer (regenerated into the layers below it; the rest stays)

In [ ]:
target = eng.object_order(scene)[-1]
eng.update_layer(scene, target.id, prompt='a skeleton playing a fiddle on the right side of the fire', progress=print)
show(scene, 're-prompted ' + target.name)

## 4. Story event: a new element generated into the current scene

In [ ]:
eng.add_event_layer(scene, 'forest fire', 'the pine trees on the left are on fire, bright flames and smoke', progress=print)
show(scene, 'after story event')

## 5. Move / resize a layer (no regeneration) and reorder

In [ ]:
el = eng.object_order(scene)[0]
eng.transform_layer(scene, el.id, dx=-0.1, dy=0.05, scale=1.2)
show(scene, f'moved and resized {el.name}')

In [ ]:
shutil.make_archive('/kaggle/working/scenes_out', 'zip', S.scenes_dir)
print('Download /kaggle/working/scenes_out.zip from the Output panel; unzip into prototype/scenes to open it in the local UI')